## 01

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import re
from konlpy.tag import Okt

In [2]:
# 1. 데이터 준비
data = {
    'reviews': [
        '이 영화 정말 좋아 최고야',
        '시간 아까운 쓰레기 영화',
        '배우들 연기가 너무 훌륭해요',
        '스토리가 지루하고 뻔하다',
        '인생 최고의 명작입니다 추천',
        '돈 주고 보기 아까운 졸작'
    ],
    'ratings': [1, 0, 1, 0, 1, 0]
}

df = pd.DataFrame(data)

In [3]:
# 2. 전처리 + 형태소 분석 토큰화
def clean_data(text):
    return re.sub(r'[^가-힣\s]', '', text)

df['clean'] = df['reviews'].apply(lambda x : clean_data(x))

okt = Okt()
def pos_tokenize(text):
    return [ word for word, pos in okt.pos(text)  
            if pos in ['Noun', 'Verb', 'Adjective'] and len(word) >=2 ]

df['tokens'] = df['clean'].apply(lambda x : pos_tokenize(x))

In [ ]:
vocab = {
    '<PAD>' : 0,
    '<UNK>' : 1
}
def make_vocab(text):
    for token in text:
        if token not in vocab:
            vocab[token] = len(vocab)

df['tokens'].apply(lambda x : make_vocab(x))
VOCAB_SIZE = len(vocab)

In [6]:
MAX_LEN = 5

def text_to_sequence(vocab, text, maxlen):
    seq = [ vocab.get(word, vocab['<UNK>']) for word in text ]
    if len(seq) < maxlen:
        seq += [vocab['<PAD>']] * ( MAX_LEN - len(seq) ) 
    
    return seq[:MAX_LEN]

df['sequence'] = df['tokens'].apply(lambda x : text_to_sequence(vocab, x, MAX_LEN))

In [7]:
df.head()

,reviews,ratings,clean,tokens,sequence
0,이 영화 정말 좋아 최고야,1,이 영화 정말 좋아 최고야,"[영화, 정말, 좋아, 최고]","[2, 3, 4, 5, 0]"
1,시간 아까운 쓰레기 영화,0,시간 아까운 쓰레기 영화,"[시간, 아까운, 쓰레기, 영화]","[6, 7, 8, 2, 0]"
2,배우들 연기가 너무 훌륭해요,1,배우들 연기가 너무 훌륭해요,"[배우, 연기, 훌륭해요]","[9, 10, 11, 0, 0]"
3,스토리가 지루하고 뻔하다,0,스토리가 지루하고 뻔하다,"[스토리, 지루하고, 뻔하다]","[12, 13, 14, 0, 0]"
4,인생 최고의 명작입니다 추천,1,인생 최고의 명작입니다 추천,"[인생, 최고, 명작, 입니다, 추천]","[15, 5, 16, 17, 18]"


In [8]:
class ReviewDataset(Dataset):
    def __init__(self, sequences, labels):
        self.x = torch.LongTensor(sequences)
        self.y = torch.FloatTensor(labels)
    def __len__(self):
        return len(self.x)
    def __getitem__(self,idx):
        return self.x[idx], self.y[idx]
    
dataset = ReviewDataset(
    df['sequence'].tolist(),
    df['ratings'].tolist()
)

dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

In [ ]:
# TextCNN 모델
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super.__init__()
        self.embedding = nn.Embedding(
            vocab_size = vocab_size,
            embed_dim = embed_dim,
            padding_idx = 0
        )
        

## 03

In [35]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from gensim.models import Word2Vec

In [36]:
# 데이터 준비
data = {
    'reviews': [
        '이 영화 정말 좋아 최고야',
        '시간 아까운 쓰레기 영화',
        '배우들 연기가 너무 훌륭해요',
        '스토리가 지루하고 뻔하다',
        '인생 최고의 명작입니다 추천',
        '돈 주고 보기 아까운 졸작'
    ],
    'ratings': [5, 1, 4, 2, 5, 1]
}

df = pd.DataFrame(data)

df['ratings'] = df['ratings'].apply(lambda x : 1 if x >= 4 else 0 )

In [37]:
# 토큰화
def tokenize(text):
    return text.split()

tokenized_sentences = [
    tokenize(review)
    for review in df['reviews']
]

In [38]:
# Word2Vec 학습
EMBED_DIM = 100

w2v_model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=EMBED_DIM,          # 100차원 벡터   
    window=3,                       # 앞, 뒤 3단어 참고
    min_count=1,                    # 1번 이상 등장 단어 학습, (보통은 좀더 큼)
    workers=4                       # CPU코어 4개를 동시에 써서 학습 속도 높임
)

print("Word2Vec 사전학습 완료")
# 학습이 끝나면 w2v_model.wv['좋아'] 이런 식으로 단어 벡터를 꺼낼 수 있음

Word2Vec 사전학습 완료


In [39]:
# 단어 사전 만들기
vocab = {
    '<PAD>' : 0,
    '<UNK>' : 1
}

for sentence in tokenized_sentences:
    for word in sentence:
        if word not in vocab:
            vocab[word] = len(vocab)

VOCAB_SIZE = len(vocab)

In [40]:
# 문장 -> 숫자 시퀀스 변환 + 패딩
MAX_LEN = 10

def text_to_sequence(text, vocab, max_len):
    seq = [
        vocab.get(word, vocab['<UNK>'])
        for word in tokenize(text)
    ]
    if len(seq) < max_len:
        seq += [vocab['<PAD>']] * (max_len - len(seq))
    return seq[:max_len]

df['sequence'] = df['reviews'].apply(
    lambda x: text_to_sequence(x, vocab, MAX_LEN))

In [41]:
# Embedding Matrix 만들기
embedding_matrix = np.random.normal(
    scale=0.01,
    size=(VOCAB_SIZE, EMBED_DIM)
)
# (VOCAB_SIZE, EMBED_DIM) 크기의 행렬을 아주 작은 랜덤값으로 채우기
# 행 하나가 단어 하나의 벡터

embedding_matrix[0] = np.zeros((EMBED_DIM,))
# <PAD>는 의미 없는 패딩이라 0벡터로 설정

for word, idx in vocab.items():
    if word in ['<PAD>', '<UNK>']:
        continue
    if word in w2v_model.wv:
        embedding_matrix[idx] = w2v_model.wv[word]
# 사전의 모든 단어를 순회하면서 Word2Vec이 학습한 벡터를 행렬에 복사

pretrained_weight = torch.FloatTensor(embedding_matrix) # numpy행렬을 Pytorch 텐서로 변환

In [42]:
# Dataset 만들기
class ReviewDataset(Dataset):
    def __init__(self, sequences, labels):
        self.x = torch.LongTensor(sequences)
        self.y = torch.FloatTensor(labels)

    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]
    
dataset = ReviewDataset(
    df['sequence'].tolist(),
    df['ratings'].tolist()
)

dataloader = DataLoader(
    dataset,
    batch_size = 2,
    shuffle=True
)

In [43]:
# TextCNN 모델 정의
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, filter_sizes, num_filters, pretrained_weight=None, freeze_emb=False):
        super().__init__()
        # 단어 숫자를 벡터로 바꿔주는 레이어
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )

        # WordVec 벡터를 Embedding 레이어데 덮어씀 => 전이 학습
        if pretrained_weight is not None:
            self.embedding.weight.data.copy_(pretrained_weight)
            self.embedding.weight.requires_grad = not freeze_emb

        self.convs = nn.ModuleList([
            nn.Conv2d(
                in_channels=1,              # 흑백 이미지처럼 채널이 1개
                out_channels=num_filters,   # 필터 개수, 20개면 각 크기마다 20가지 패턴을 탐지
                kernel_size=(fs, embed_dim) # 필터 높이는 단어 수, 너비는 embed_dim전체 => 단어 벡터 전체를 한 번에 봄
            )
            for fs in filter_sizes          # filter_sizes=[2,3,4]이면 크기가 2, 3, 4짜리 필터를 각각 만듬
        ])

        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(len(filter_sizes) * num_filters, 1)

    # TextCNN forward 함수
    def forward(self, x):
        x = self.embedding(x)
        x = x.unsqueeze(1)

        pooled_outputs = []
        for conv in self.convs:
            c = F.relu(conv(x))
            c = c.squeeze(3)
            p = F.max_pool1d(c, kernel_size=c.size(2))
            p = p.squeeze(2)
            pooled_outputs.append(p)

        x = torch.cat(pooled_outputs, dim=1)
        x = self.dropout(x)
        logits = self.fc(x)
        return logits.squeeze(1)

In [44]:
# 모델 생성 및 LOss/Optimizer 설정
FILTER_SIZES = [2, 3, 4]
NUM_FILTERS = 20

model = TextCNN(
    vocab_size = VOCAB_SIZE,
    embed_dim = EMBED_DIM,
    filter_sizes = FILTER_SIZES,
    num_filters=NUM_FILTERS,
    pretrained_weight = pretrained_weight,
    freeze_emb = False
)

print(model)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

TextCNN(
  (embedding): Embedding(25, 100, padding_idx=0)
  (convs): ModuleList(
    (0): Conv2d(1, 20, kernel_size=(2, 100), stride=(1, 1))
    (1): Conv2d(1, 20, kernel_size=(3, 100), stride=(1, 1))
    (2): Conv2d(1, 20, kernel_size=(4, 100), stride=(1, 1))
  )
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=60, out_features=1, bias=True)
)


In [45]:
# 학습
EPOCHS = 10

print("\n학습 시작\n")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch_x, batch_y in dataloader:
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch: {epoch+1:02d} Loss: {avg_loss:.4f}")

print("\n학습완료")



학습 시작

Epoch: 01 Loss: 0.6903
Epoch: 02 Loss: 0.6932
Epoch: 03 Loss: 0.6913
Epoch: 04 Loss: 0.6889
Epoch: 05 Loss: 0.6813
Epoch: 06 Loss: 0.6765
Epoch: 07 Loss: 0.6816
Epoch: 08 Loss: 0.6689
Epoch: 09 Loss: 0.6733
Epoch: 10 Loss: 0.6807

학습완료


In [46]:
# 예측 함수 + 테스트
def predict(text):
    model.eval()
    sequence = text_to_sequence(text, vocab, MAX_LEN)
    x = torch.LongTensor(sequence).unsqueeze(0)
    with torch.no_grad():
        logits = model(x)
        prob = torch.sigmoid(logits)
        pred = (prob >= 0.5).float()

    print(f"\n리뷰: {text}")
    print(f"긍정확률 : {prob.item():.4f}")
    if pred.item() == 1:
        print("예측: 긍정")
    else:
        print("예측: 부정")

# 테스트
predict("배우 연기가 정말 훌륭하다")
predict("돈 아까운 최악의 영화")
predict("이 영화는 재미있고 추천할 만 합니다")
predict("재미없는 영화")


리뷰: 배우 연기가 정말 훌륭하다
긍정확률 : 0.5332
예측: 긍정

리뷰: 돈 아까운 최악의 영화
긍정확률 : 0.5172
예측: 긍정

리뷰: 이 영화는 재미있고 추천할 만 합니다
긍정확률 : 0.5285
예측: 긍정

리뷰: 재미없는 영화
긍정확률 : 0.5272
예측: 긍정
